# Chroma
- Chroma 공식 https://cookbook.chromadb.dev/ 
- Chroma-admin  https://github.com/flanker/chromadb-admin.git

In [5]:
from langchain_community.document_loaders import PyMuPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings 
from langchain_chroma import Chroma 

In [7]:
loader = PyMuPDFLoader("./data/2024_KB_부동산_보고서_최종.pdf")
pages = loader.load()
print(f"청크의 수 : {len(pages)}" )

청크의 수 : 84


In [10]:
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_spliter.split_documents(pages)
print(f"분할된 청크의 수 : {len(splits)}" )

분할된 청크의 수 : 135


In [11]:
chunk_lengths = [len(chunk.page_content) for chunk in splits]
max_length = max(chunk_lengths)
min_length = min(chunk_lengths)
avg_length = sum(chunk_lengths) / len(chunk_lengths)

print(f"청크의 최대 길이 {max_length}")
print(f"청크의 최소 길이 {min_length}")
print(f"청크의 평균 길이 {avg_length}")

청크의 최대 길이 999
청크의 최소 길이 57
청크의 평균 길이 675.6148148148148


In [17]:
import chromadb

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    tenant="default_tenant",
    database="default_database",
)

print(client.heartbeat())

1782365135965955879


In [24]:
embedding_function = OpenAIEmbeddings(
    model="bge-m3",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    check_embedding_ctx_length=False,
)

try:
    client.delete_collection("real_estate")
except Exception:
    pass 

vector_db = Chroma.from_documents(
    documents=splits, 
    embedding=embedding_function,
    collection_name="real_estate",
    client = client
)

In [31]:
print(f"컬렉션:{client.list_collections()}")

컬렉션:[Collection(name=real_estate)]


In [25]:
print(f"문서의 수 :{vector_db._collection.count()}")

문서의 수 :135


In [34]:
question = "수도권 주택 매매 전망"
top_three_docs = vector_db.similarity_search(question, k=2)
for i, doc in enumerate(top_three_docs, 1):
    print(f"문서 : {i}")
    print(f"내용 {doc.page_content[:150]}]")
    print(f"메타데이터 : {doc.metadata}")
    print("*******")


문서 : 1
내용 40 
2024 KB 부동산 보고서: 수도권 주택시장 점검 
 
 
주택 매매가격은 2023년 상반기 이후 회복세를 보였으나 상승폭이 크지 않았으며 12월 다시 하락세
로 전환되었다. 2023년 연간 주택 매매가격은 4.64% 하락하면서 2022년(1.83% 하락)에 ]
메타데이터 : {'format': 'PDF 1.5', 'creator': 'Microsoft® Word 2016', 'creationDate': "D:20240304153001+09'00'", 'producer': 'Microsoft® Word 2016', 'total_pages': 84, 'keywords': '', 'file_path': './data/2024_KB_부동산_보고서_최종.pdf', 'trapped': '', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'page': 46, 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'title': 'Morning Meeting', 'creationdate': '2024-03-04T15:30:01+09:00', 'modDate': "D:20240304153001+09'00'", 'subject': ''}
*******
문서 : 2
내용 28 
2024 KB 부동산 보고서: 주택시장 설문조사 
1) 2024년 주택시장 전망 
■ 주택 매매가격, 2024년에도 하락 전망 우세한 가운데 상승 전망 2023년 대비 증가 
부동산시장 전문가와 공인중개사, 자산관리전문가(PB)를 대상으로 한 설문조사 결과, 2]
메타데이터 : {'creator': 'Microsoft® Word 2016', 'subject': '', 'trapped': '', 'author': '손은경', 'modDate': "D:20240304153001+09'00'", 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'format': 'PDF 1.5

In [37]:
question  = "수도권 주택 매매 전망" 

top_three_docs = vector_db.similarity_search_with_relevance_scores(question, k=3)

for i, doc in enumerate(top_three_docs, 1):
    print(f"문서 : {i}")
    print(f"유사 점수 {doc[1]}")
    print(f"내용 {doc[0].page_content[:150]}]")
    print(f"메타데이터 : {doc[0].metadata}")
    print("*******")

문서 : 1
유사 점수 0.5509815370922928
내용 40 
2024 KB 부동산 보고서: 수도권 주택시장 점검 
 
 
주택 매매가격은 2023년 상반기 이후 회복세를 보였으나 상승폭이 크지 않았으며 12월 다시 하락세
로 전환되었다. 2023년 연간 주택 매매가격은 4.64% 하락하면서 2022년(1.83% 하락)에 ]
메타데이터 : {'subject': '', 'creationDate': "D:20240304153001+09'00'", 'creationdate': '2024-03-04T15:30:01+09:00', 'total_pages': 84, 'creator': 'Microsoft® Word 2016', 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'format': 'PDF 1.5', 'file_path': './data/2024_KB_부동산_보고서_최종.pdf', 'producer': 'Microsoft® Word 2016', 'title': 'Morning Meeting', 'keywords': '', 'moddate': '2024-03-04T15:30:01+09:00', 'trapped': '', 'modDate': "D:20240304153001+09'00'", 'page': 46, 'author': '손은경'}
*******
문서 : 2
유사 점수 0.5449613713786268
내용 28 
2024 KB 부동산 보고서: 주택시장 설문조사 
1) 2024년 주택시장 전망 
■ 주택 매매가격, 2024년에도 하락 전망 우세한 가운데 상승 전망 2023년 대비 증가 
부동산시장 전문가와 공인중개사, 자산관리전문가(PB)를 대상으로 한 설문조사 결과, 2]
메타데이터 : {'format': 'PDF 1.5', 'total_pages': 84, 'page': 34, 'creationDate': "D:20240304153001+09'00'", 'keywords': '', 'creationdate': '202